In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


In [2]:
# Load the dataset safely
df = pd.read_csv("spam.csv", encoding='latin-1')

# Keep only the first two populated columns and rename them cleanly
df = df[['v1', 'v2']]
df.columns = ['Category', 'Message']

print("Dataset Cleaned Columns:", df.columns.tolist())
print(df.head(3))


Dataset Cleaned Columns: ['Category', 'Message']
  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...


In [3]:
# Map category text strings to binary integers
df['Label'] = df['Category'].map({'spam': 1, 'ham': 0})

print("Class Distributions:")
print(df['Category'].value_counts())


Class Distributions:
Category
ham     4825
spam     747
Name: count, dtype: int64


In [4]:
X = df['Message']
y = df['Label']

# 80% training set, 20% validation test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training messages count: {len(X_train)}")
print(f"Testing messages count: {len(X_test)}")


Training messages count: 4457
Testing messages count: 1115


In [5]:
# Initialize vectorizer pipeline
vectorizer = TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)

# Generate vocabulary feature space matrices
X_train_features = vectorizer.fit_transform(X_train)
X_test_features = vectorizer.transform(X_test)


In [6]:
# Initialize and train Logistic Regression model
model = LogisticRegression()
model.fit(X_train_features, y_train)

print("Text classifier engine training complete!")


Text classifier engine training complete!


In [7]:
# Evaluate model output predictions
y_pred = model.predict(X_test_features)

print(f"Model Detection Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham (Valid)', 'Spam (Junk)']))


Model Detection Accuracy: 96.77%

Detailed Classification Report:
              precision    recall  f1-score   support

 Ham (Valid)       0.96      1.00      0.98       966
 Spam (Junk)       1.00      0.76      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115



In [8]:
# Define custom incoming text messages to test live filtering
new_test_emails = [
    "URGENT! Call 09061701461 now to claim your £900 prize reward immediately! Code KL341.",
    "Hey, are we still meeting up for dinner tonight? Let me know if this time works."
]

# Transform text to vectors matching original vocab rules
new_email_features = vectorizer.transform(new_test_emails)

# Predict binary values
predictions = model.predict(new_email_features)

for email, pred in zip(new_test_emails, predictions):
    status = "SPAM 🚨" if pred == 1 else "HAM (Legitimate) ✅"
    print(f"\nMessage: {email}\nResult: {status}")



Message: URGENT! Call 09061701461 now to claim your £900 prize reward immediately! Code KL341.
Result: SPAM 🚨

Message: Hey, are we still meeting up for dinner tonight? Let me know if this time works.
Result: HAM (Legitimate) ✅
